# BME 574 - Lecture 6 companion
# Autoencoders and the latent space

Build an autoencoder, look at the space it learns, then try to **use** that
space to make new digits. The last part fails, and how it fails is the whole
reason the next lecture exists.

What you will do:

1. Train an autoencoder on MNIST. No labels anywhere.
2. Check it against PCA at the same bottleneck size - this is Project 1 with
   the straight lines taken out.
3. **Draw the entire latent space** and explore it: click around, decode points,
   interpolate between digits.
4. Measure how much of that space is actually usable. (Spoiler: about 44%.)
5. Try three ways to generate new digits, and watch all three come up short.
6. Work out exactly which line of the loss function is responsible.

**Runtime.** About 6 minutes on two CPU cores, most of it in section 1. A GPU is
used automatically if you have one.

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

torch.manual_seed(0); np.random.seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEV)

transform = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.MNIST(root='./data', train=True,
                                       download=True, transform=transform)
test_set  = torchvision.datasets.MNIST(root='./data', train=False,
                                       download=True, transform=transform)

N_TRAIN = 20000                      # a subset keeps this fast; use 60000 if you like
X  = (train_set.data[:N_TRAIN].float() / 255).reshape(-1, 784).to(DEV)
Y  =  train_set.targets[:N_TRAIN].clone().to(DEV)
Xt = (test_set.data.float() / 255).reshape(-1, 784).to(DEV)
Yt =  test_set.targets.clone().to(DEV)
print('train', tuple(X.shape), '| test', tuple(Xt.shape))

def show(imgs, n=10, title='', figsize=None):
    a = imgs.detach().cpu().reshape(-1, 28, 28).numpy()[:n]
    fig, ax = plt.subplots(1, n, figsize=figsize or (1.1 * n, 1.4))
    for i in range(n):
        ax[i].imshow(a[i], cmap='gray', vmin=0, vmax=1); ax[i].axis('off')
    if title: fig.suptitle(title, y=1.1, fontsize=11)
    plt.show()

show(X, title='MNIST. No labels will be used to train anything in sections 1-5.')

---
## 1. The autoencoder

Two networks trained together:

- an **encoder** $e_\phi$ that squeezes 784 pixels down to $d_z$ numbers,
- a **decoder** $d_\theta$ that expands those numbers back to 784 pixels.

The loss compares the output to the **input**. There is no $y$ anywhere:

$$ \mathcal{L} \;=\; \frac{1}{N}\sum_{n=1}^{N} \left\| x_n - d_\theta\!\left(e_\phi(x_n)\right)\right\|^2 $$

The middle layer is narrow **on purpose**. If it were 784 wide, the network
could learn the identity function and learn nothing at all. Being forced to
throw information away is what forces it to decide what matters.

In [ ]:
class AE(nn.Module):
    def __init__(self, d_z=2, d_hid=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(784, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, d_z))          # <- the bottleneck
        self.dec = nn.Sequential(nn.Linear(d_z, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, 784), nn.Sigmoid())
        self.d_z = d_z

    def forward(self, x):
        return self.dec(self.enc(x))

def train_ae(d_z, epochs=25, seed=0, lr=1e-3, log=False):
    torch.manual_seed(seed)
    m = AE(d_z).to(DEV)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    for ep in range(epochs):
        perm = torch.randperm(len(X), generator=g).to(DEV)
        tot = 0.0
        for i in range(0, len(X), 128):
            xb = X[perm[i:i + 128]]
            loss = ((m(xb) - xb) ** 2).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(xb)
        if log and (ep % 8 == 0 or ep == epochs - 1):
            print('   epoch %2d   reconstruction error %7.2f' % (ep, tot / len(X)))
    m.eval(); return m

def recon_error(m, A, bs=2000):
    with torch.no_grad():
        return float(torch.cat([((m(A[i:i+bs]) - A[i:i+bs]) ** 2).sum(1)
                                for i in range(0, len(A), bs)]).mean())

print('training a 2-dimensional autoencoder (small enough to draw)')
t0 = time.time()
ae2 = train_ae(2, epochs=30, log=True)
print('   done in %.0f s | held-out error %.2f' % (time.time() - t0, recon_error(ae2, Xt)))

---
## 2. You have already built one of these

Make the encoder and decoder single **linear** layers and the best solution is
**PCA**. So an autoencoder is PCA with the straight lines taken out. Does the
non-linearity earn its keep?

In [ ]:
mean_x = X.mean(0, keepdim=True)
U, S, Vt = torch.linalg.svd(X - mean_x, full_matrices=False)
V = Vt.T

def pca_error(k):
    Z = (Xt - mean_x) @ V[:, :k]
    return float(((mean_x + Z @ V[:, :k].T - Xt) ** 2).sum(1).mean())

DIMS = [2, 5, 10, 20, 50]
models, rows = {2: ae2}, []
for k in DIMS:
    if k not in models:
        models[k] = train_ae(k, epochs=20)
    rows.append((k, pca_error(k), recon_error(models[k], Xt)))
    print('d_z=%-3d  PCA %7.2f   autoencoder %7.2f' % rows[-1])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(DIMS, [r[1] for r in rows], 'o--', color='#808080', lw=2, label='PCA (linear)')
ax.plot(DIMS, [r[2] for r in rows], 'o-', color='#1F3864', lw=2.4, label='autoencoder')
ax.set_xscale('log'); ax.set_xticks(DIMS); ax.set_xticklabels(DIMS)
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_xlabel('bottleneck size'); ax.set_ylabel('reconstruction error')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

k = 10
with torch.no_grad():
    r_ae = models[k](Xt[:8])
Zp = (Xt[:8] - mean_x) @ V[:, :k]
r_pca = (mean_x + Zp @ V[:, :k].T).clamp(0, 1)
for imgs, lab in [(Xt[:8], 'original'), (r_pca, 'PCA, k=10'), (r_ae, 'autoencoder, d_z=10')]:
    show(imgs, n=8, title=lab)

The gap is largest at a tight bottleneck and closes as you add components. That
is the signature of curvature: the set of digit images is curved, a plane
through it fits badly, and with enough components a linear model can cover the
curve piecewise.

PCA still has real advantages - closed form, deterministic, interpretable axes.
It just is not the most accurate compressor here.

---
## 3. The map

With $d_z = 2$ the whole latent space fits on a screen. Encode every test digit
and plot it.

**The colours are the true labels, added afterwards for this plot. The
autoencoder never saw a single one.**

In [ ]:
with torch.no_grad():
    Z2 = ae2.enc(Xt)
Z = Z2.cpu().numpy(); lab = Yt.cpu().numpy()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sc = ax.scatter(Z[:, 0], Z[:, 1], c=lab, s=4, cmap='tab10', alpha=.6)
ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')
ax.set_title('every test digit, encoded to two numbers')
cb = plt.colorbar(sc, ax=ax, ticks=range(10)); cb.set_label('true digit')
plt.tight_layout(); plt.show()

print('code range:  z1 [%.1f, %.1f]   z2 [%.1f, %.1f]'
      % (Z[:, 0].min(), Z[:, 0].max(), Z[:, 1].min(), Z[:, 1].max()))
print('\nThings to look for before you read on:')
print('  - which digits sit next to each other, and whether that makes sense')
print('  - which pair overlaps almost completely')
print('  - how tight the 1s are compared with everything else')
print('  - the SCALE of the axes. Nothing fixed that range. Remember it.')

### Read the map

Compression alone, with no supervision, discovered that there are roughly ten
kinds of thing here and which of them look alike. The mistakes it makes are the
mistakes a person makes: 4 and 9 overlap heavily, 3/5/8 share a region, 1s form
a tight spike because they vary the least.

Now note the **axis range**. The codes run over tens of units, and nothing in
the loss chose that. Retrain with a different seed and you get a different
range. There is no such thing as a "typical" value of $z$ here - which is going
to matter a great deal in section 5.

In [ ]:
# Explore it yourself. Change these two numbers and re-run the cell.
z1, z2 = 0.0, 0.0

with torch.no_grad():
    img = ae2.dec(torch.tensor([[z1, z2]], dtype=torch.float32, device=DEV))
    d_near = torch.cdist(torch.tensor([[z1, z2]], dtype=torch.float32, device=DEV),
                         ae2.enc(X)).min().item()
fig, ax = plt.subplots(1, 2, figsize=(8.5, 3.6))
ax[0].scatter(Z[:, 0], Z[:, 1], c=lab, s=3, cmap='tab10', alpha=.35)
ax[0].plot(z1, z2, 'kx', ms=14, mew=3)
ax[0].set_title('you are here', fontsize=10); ax[0].set_xlabel('$z_1$'); ax[0].set_ylabel('$z_2$')
ax[1].imshow(img.cpu().reshape(28, 28), cmap='gray', vmin=0, vmax=1); ax[1].axis('off')
ax[1].set_title('decoded (%.2f from the nearest training code)' % d_near, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# The decoder as a chart: decode a regular grid and lay the images out.
R, lo, hi = 20, Z.min(0), Z.max(0)
gx = np.linspace(lo[0], hi[0], R); gy = np.linspace(hi[1], lo[1], R)
GY, GX = np.meshgrid(gy, gx, indexing='ij')
G = torch.tensor(np.stack([GX.ravel(), GY.ravel()], 1), dtype=torch.float32, device=DEV)
with torch.no_grad():
    grid_imgs = ae2.dec(G)
canvas = grid_imgs.cpu().reshape(R, R, 28, 28).numpy().transpose(0, 2, 1, 3).reshape(R*28, R*28)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(canvas, cmap='gray', extent=[lo[0], hi[0], lo[1], hi[1]], aspect='auto')
ax.set_xlabel('$z_1$'); ax.set_ylabel('$z_2$')
ax.set_title('the decoder, evaluated everywhere')
plt.tight_layout(); plt.show()

---
## 4. How much of this space is actually usable?

Looking at the chart above, some regions are clean digits and some are smears.
We need to say that with a number rather than an impression.

**A warning about instruments.** The obvious idea is to feed each decoded image
to a classifier and look at the confidence. That does not work, and §6 shows why
in detail. Distance in pixel space to the nearest real digit does not work
either - a blend of a 3 and an 8 has the right amount of ink and the right
stroke width, it just isn't a digit.

So measure the problem **where the problem is**: in the latent space.

> How isolated does a **real** code ever get? Take each training code, find its
> nearest neighbour among the others, and look at the distribution. If a point
> is further from every training code than real codes ever are from each other,
> then the decoder has never been trained anywhere near it. That is a hole, and
> it needs no judgement call.

In [ ]:
with torch.no_grad():
    Ztr = ae2.enc(X)

# how far is a real code from its nearest OTHER real code?
d_real = torch.cdist(Ztr[:3000], Ztr).kthvalue(2, dim=1).values
TYP = float(d_real.quantile(0.99))
print('distance from a real code to its nearest neighbour:')
print('   median %.3f | 99th percentile %.3f' % (d_real.median(), TYP))
print('\nSo call a point a HOLE if it is further than %.2f from every training code.' % TYP)

d_grid = torch.cdist(G, Ztr).min(1).values
covered = float((d_grid < TYP).float().mean())
print('\nFraction of the latent plane that is NOT a hole: %.1f%%' % (100 * covered))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ax[0].imshow(canvas, cmap='gray', extent=[lo[0], hi[0], lo[1], hi[1]], aspect='auto')
ax[0].set_title('what the decoder paints', fontsize=10)
im = ax[1].imshow(np.minimum(d_grid.cpu().numpy().reshape(R, R), 4 * TYP), cmap='RdYlGn_r',
                  extent=[lo[0], hi[0], lo[1], hi[1]], aspect='auto')
ax[1].scatter(Z[::7, 0], Z[::7, 1], s=1.2, c='k', alpha=.3)
ax[1].set_title('distance to the nearest training code\n(red = the decoder was never trained here)',
                fontsize=10)
for a in ax:
    a.set_xlabel('$z_1$'); a.set_ylabel('$z_2$')
plt.colorbar(im, ax=ax[1], fraction=.046); plt.tight_layout(); plt.show()

Compare the two panels. The clean digits are exactly where the black dots are.
The smears are exactly where they are not. And note that the boundary **between**
two digit clusters is red, not yellow: the space between two classes is not a
smooth blend, it is a gap.

---
## 5. Three attempts to generate

Now the thing we actually wanted. Pick a $z$, decode it, get a new digit.

In [ ]:
g = torch.Generator().manual_seed(5)

# attempt 1 -- the naive thing
z_naive = torch.randn(10, 2, generator=g).to(DEV)
# attempt 2 -- fix the scale: uniform over the box the codes occupy
lo_t = torch.tensor(lo, dtype=torch.float32, device=DEV)
hi_t = torch.tensor(hi, dtype=torch.float32, device=DEV)
z_box = lo_t + (hi_t - lo_t) * torch.rand(10, 2, generator=g).to(DEV)
# attempt 3 -- fit a Gaussian to the codes we actually have
z_fit = Z2.mean(0) + Z2.std(0) * torch.randn(10, 2, generator=g).to(DEV)

with torch.no_grad():
    for z, nm in [(z_naive, 'attempt 1:  z ~ N(0, I)'),
                  (z_box,   'attempt 2:  uniform over the code box'),
                  (z_fit,   'attempt 3:  a Gaussian fitted to the codes')]:
        show(ae2.dec(z), title=nm)
show(Xt, title='real test digits, for comparison')

### Be honest about what you are seeing

Attempt 1 is bad for an almost silly reason: the codes live at a scale of tens,
so a standard normal samples a tiny blob near the origin that happens to be in
the middle of nowhere. **The latent space has no units.**

Attempts 2 and 3 are better. In fact at a 2-D bottleneck, attempt 3 nearly gets
away with it - the data fills most of a small plane, so a Gaussian over it lands
on something reasonable more often than not.

If this were the whole story you would be entitled to ask what the fuss is
about. So run the same experiment at the bottleneck size you would actually use.

In [ ]:
print('%-8s %14s %14s %16s' % ('d_z', 'sampled code', 'real code', '% in a hole'))
hole_frac = {}
for k in DIMS[:4]:
    m = models[k]
    with torch.no_grad():
        Zk = m.enc(X)
    mu_k, sd_k = Zk.mean(0), Zk.std(0)
    gq = torch.Generator().manual_seed(7)
    zs = mu_k + sd_k * torch.randn(2000, k, generator=gq).to(DEV)
    d_s = torch.cdist(zs, Zk).min(1).values
    d_r = torch.cdist(Zk[:2000], Zk).kthvalue(2, dim=1).values
    hole_frac[k] = float((d_s > d_r.quantile(0.99)).float().mean())
    print('%-8d %14.2f %14.2f %15.1f%%'
          % (k, d_s.median(), d_r.median(), 100 * hole_frac[k]))

fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.plot(list(hole_frac), [100 * v for v in hole_frac.values()], 'o-',
        color='#9C2B2B', lw=2.4, ms=7)
ax.set_xscale('log'); ax.set_xticks(list(hole_frac)); ax.set_xticklabels(list(hole_frac))
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_xlabel('bottleneck size'); ax.set_ylabel('% of sampled codes in a hole')
ax.set_ylim(0, 100); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

with torch.no_grad():
    m20 = models[20]
    Z20 = m20.enc(X)
    gq = torch.Generator().manual_seed(5)
    z20 = Z20.mean(0) + Z20.std(0) * torch.randn(10, 20, generator=gq).to(DEV)
    show(m20.dec(z20), title='the SAME sampling scheme at a bottleneck of 20')

**This is a volume problem.** In two dimensions the data occupies a decent
fraction of the box around it. In twenty dimensions it is a vanishing sliver,
and almost everything you sample lands outside. So the failure is invisible in
the picture we can draw and severe in the dimensions we actually use.

No amount of cleverness at *sampling* time fixes a space that was never
organised at *training* time.

In [ ]:
# Attempt 4: interpolate. Both endpoints are real digits -- what is in between?
i3 = int((Yt == 3).nonzero()[0]); i8 = int((Yt == 8).nonzero()[0])
STEPS = 11
t = torch.linspace(0, 1, STEPS, device=DEV).unsqueeze(1)
with torch.no_grad():
    za, zb = ae2.enc(Xt[i3:i3+1]), ae2.enc(Xt[i8:i8+1])
    zline = (1 - t) * za + t * zb
    show(ae2.dec(zline), n=STEPS, title='interpolating in the LATENT space: a 3 to an 8')
    show((1 - t) * Xt[i3:i3+1] + t * Xt[i8:i8+1], n=STEPS,
         title='interpolating in PIXELS, for comparison')
    d_path = torch.cdist(zline, Ztr).min(1).values.cpu().numpy()

fig, ax = plt.subplots(figsize=(6.4, 2.6))
ax.plot(range(STEPS), d_path, 'o-', color='#1F3864', lw=2.2)
ax.axhline(TYP, color='#2E6B4F', ls=':', lw=2, label='as isolated as a real code gets')
ax.fill_between(range(STEPS), TYP, d_path, where=(d_path > TYP), color='#9C2B2B', alpha=.2)
ax.set_xlabel('step from the 3 to the 8'); ax.set_ylabel('distance to the\nnearest real code')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print('The straight line between two real digits leaves the data. Max distance %.2f'
      % d_path.max())

---
## 6. Why I did not use a classifier as the instrument

This is a short detour and it is worth more than a detour, because it is a
mistake that will otherwise appear in your own projects.

Train an accurate MNIST classifier. Feed it images decoded from the **empty**
half of the latent plane - places we have just established the decoder was never
trained. Then look at what it says.

In [ ]:
class Clf(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(32 * 49, 10))

    def forward(self, x):
        return self.net(x.reshape(-1, 1, 28, 28))

torch.manual_seed(0)
clf = Clf().to(DEV); opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
gg = torch.Generator().manual_seed(0)
for ep in range(3):
    perm = torch.randperm(len(X), generator=gg).to(DEV)
    for i in range(0, len(X), 128):
        b = perm[i:i+128]
        F.cross_entropy(clf(X[b]), Y[b]).backward(); opt.step(); opt.zero_grad()
clf.eval()
with torch.no_grad():
    print('classifier test accuracy %.4f' % float((clf(Xt).argmax(1) == Yt).float().mean()))

# ten points spread across the empty half of the plane
gq = torch.Generator().manual_seed(21)
cand = lo_t + (hi_t - lo_t) * torch.rand(4000, 2, generator=gq).to(DEV)
empty = cand[torch.cdist(cand, Ztr).min(1).values > 2 * TYP]
sel = [int(torch.argmax(torch.cdist(empty, Ztr).min(1).values))]
for _ in range(9):                                   # farthest-point, so they differ
    sel.append(int(torch.argmax(torch.cdist(empty, empty[sel]).min(1).values)))
with torch.no_grad():
    imgs = ae2.dec(empty[sel])
    p = torch.softmax(clf(imgs), 1)
conf, pred = p.max(1)

fig, ax = plt.subplots(1, 10, figsize=(11, 1.9))
for j in range(10):
    ax[j].imshow(imgs[j].cpu().reshape(28, 28), cmap='gray', vmin=0, vmax=1); ax[j].axis('off')
    ax[j].set_title('"%d"\n%.0f%% sure' % (pred[j], 100 * conf[j]), fontsize=9, color='#9C2B2B')
plt.suptitle('ten points from the empty half of the plane', y=1.25); plt.show()
print('mean confidence %.1f%%, max %.1f%%' % (100 * conf.mean(), 100 * conf.max()))

with torch.no_grad():
    c_all = torch.softmax(clf(grid_imgs), 1).max(1).values
print('\nAcross the whole grid the classifier is >90%% confident about %.1f%% of it,'
      % (100 * float((c_all > 0.9).float().mean())))
print('including the %.0f%% of the plane that is a hole.' % (100 * (1 - covered)))

A softmax over ten classes answers "**which** of these ten", conditional on it
being one of them. It is not a statement about whether the input belongs to the
problem at all. "None of the above" is not one of the options, and the
probabilities must sum to one, so the mass has to go somewhere.

Feed a 97%-accurate MNIST classifier a photograph of a cat and it will tell you,
confidently, which digit the cat is.

This is why "my classifier was confident" is never, by itself, evidence of
anything - and it is the reason Lecture 8 spends an hour on detecting inputs
that do not belong.

---
## 7. The code is still a good description

Before we condemn the autoencoder: everything above is about **generating**.
As a way of **describing** data it works well. Use the codes as features and
read out digit identity with a nearest-neighbour classifier.

The labels are used only to read the representation out, never to learn it.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
m20 = models[20]
with torch.no_grad():
    A = m20.enc(X).cpu().numpy(); B = m20.enc(Xt).cpu().numpy()
P_tr = ((X - mean_x) @ V[:, :20]).cpu().numpy(); P_te = ((Xt - mean_x) @ V[:, :20]).cpu().numpy()
yv, yq = Y.cpu().numpy(), Yt[:3000].cpu().numpy()

print('%-28s %14s' % ('representation', '10-NN accuracy'))
for nm, a, b in [('autoencoder code (20-D)', A, B),
                 ('PCA (20-D)', P_tr, P_te),
                 ('raw pixels (784-D)', X.cpu().numpy(), Xt.cpu().numpy())]:
    acc = float((KNeighborsClassifier(10).fit(a, yv).predict(b[:3000]) == yq).mean())
    print('%-28s %14.4f' % (nm, acc))
print('\nTwenty numbers describe a digit better than all 784 pixels do.')
print('Representation learning works. Generation is what does not.')

---
## 8. Which line of the loss is responsible?

Look at it again:

$$ \mathcal{L} \;=\; \frac{1}{N}\sum_n \left\| x_n - d_\theta\!\left(e_\phi(x_n)\right)\right\|^2 $$

Every term is about a **round-trip**. The round-trip is still perfect if you:

- scatter the codes over a range of a million, or
- pile them into three distant clumps with nothing in between, or
- put two visually identical digits at opposite ends of the space.

Nothing in that expression mentions **where the codes should be**. The optimiser
did exactly what we asked. We asked the wrong thing.

> A model that reconstructs perfectly can still be useless as a generator.
> Those are two different jobs, and we only paid for one of them.

### What would fix it

**1. Make the encoder output a region instead of a point.** Every image claims a
small patch of the latent space rather than a single spot. Overlapping patches
leave no gaps to fall into, and the decoder is forced to produce something
sensible on a whole neighbourhood.

**2. Add a term to the loss that pulls all those regions toward one simple,
agreed distribution** - a standard normal. Now there *is* a right answer to
"what does a typical code look like", and sampling becomes well defined.

$$ e_\phi(x) \;\longrightarrow\; \mu(x),\ \sigma(x) \qquad\qquad z \sim \mathcal{N}\!\left(\mu(x),\, \sigma^2(x)\right) $$

Those two changes are the entire difference between an autoencoder and a
**variational** autoencoder. Everything in the next lecture - the probability,
the ELBO, the reparameterisation trick - is machinery for making those two
sentences trainable.

Here is a preview of what it buys. (This cell trains a VAE; it takes about a
minute. The next lecture explains every line of it.)

In [ ]:
class VAE(nn.Module):
    def __init__(self, d_z=2, d_hid=256):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(784, d_hid), nn.ReLU(),
                                  nn.Linear(d_hid, d_hid), nn.ReLU())
        self.mu = nn.Linear(d_hid, d_z); self.lv = nn.Linear(d_hid, d_z)   # a REGION
        self.dec = nn.Sequential(nn.Linear(d_z, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, d_hid), nn.ReLU(),
                                 nn.Linear(d_hid, 784))

    def encode(self, x):
        h = self.body(x); return self.mu(h), self.lv(h)

torch.manual_seed(0)
vae = VAE(2).to(DEV); opt = torch.optim.Adam(vae.parameters(), lr=1e-3)
gv = torch.Generator().manual_seed(0)
for ep in range(20):
    perm = torch.randperm(len(X), generator=gv).to(DEV)
    for i in range(0, len(X), 128):
        xb = X[perm[i:i+128]]
        mu, lv = vae.encode(xb)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)          # the region, sampled
        rec = F.binary_cross_entropy_with_logits(vae.dec(z), xb, reduction='none').sum(1)
        kl = -0.5 * (1 + lv - mu.pow(2) - lv.exp()).sum(1)           # the pull to N(0, I)
        (rec + kl).mean().backward(); opt.step(); opt.zero_grad()
vae.eval()

with torch.no_grad():
    MU, _ = vae.encode(Xt)
    Zv = MU.cpu().numpy()
    gq = torch.Generator().manual_seed(5)
    samp = torch.sigmoid(vae.dec(torch.randn(10, 2, generator=gq).to(DEV)))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
for A, a, nm in [(Z, ax[0], 'autoencoder'), (Zv, ax[1], 'variational autoencoder')]:
    a.scatter(A[:, 0], A[:, 1], c=lab, s=3, cmap='tab10', alpha=.5)
    a.set_title('%s  (range %.0f to %.0f)' % (nm, A.min(), A.max()), fontsize=11)
    a.set_xlabel('$z_1$'); a.set_ylabel('$z_2$')
plt.tight_layout(); plt.show()

show(samp, title='VAE: samples from z ~ N(0, I) -- the SAME naive scheme that failed above')

Same architecture, same data, same training time. The codes now sit in a tidy
blob of roughly unit scale around the origin, with the classes packed against
one another instead of scattered with gaps - and the naive sampling scheme that
produced nothing useful from the autoencoder produces digits.

That is what the next lecture builds, slowly and from first principles.

---

## Exercises

1. Retrain the 2-D autoencoder with a different random seed and plot the map
   again. How much does the layout change? What does that tell you about
   interpreting the axes?
2. Set the bottleneck to 784 - the same width as the input - and train. What
   reconstruction error do you get, and what has the network learned? Why is
   this the argument for the bottleneck?
3. Add Gaussian noise to the *input* only, keeping the clean image as the
   target. This is a **denoising autoencoder** (Vincent et al., ICML 2008). Does
   the latent map look better organised? Does sampling get better?
4. Measure the hole fraction as a function of training epochs at $d_z=10$. Does
   training longer make the latent space more usable, or just lower the
   reconstruction error?
5. In §5, instead of a Gaussian, fit a 10-component Gaussian mixture to the
   codes and sample from that. How much of the gap does it close? What does your
   answer say about whether this is fixable at sampling time?
6. Take the 20-D autoencoder and the 20-D VAE from §8. Compare their 10-NN probe
   accuracies. Does the VAE's tidier latent space cost it anything as a
   *description*?

## References

- Hinton & Salakhutdinov. Reducing the Dimensionality of Data with Neural
  Networks. *Science* 313(5786):504-507, 2006.
- Baldi & Hornik. Neural networks and principal component analysis: learning
  from examples without local minima. *Neural Networks* 2(1):53-58, 1989.
- Vincent, Larochelle, Bengio & Manzagol. Extracting and Composing Robust
  Features with Denoising Autoencoders. *ICML* 2008.
- Goodfellow, Bengio & Courville. *Deep Learning*, chapter 14. MIT Press, 2016.
- Kingma & Welling. An Introduction to Variational Autoencoders. *Foundations
  and Trends in Machine Learning*, 2019. arXiv:1906.02691.